# VAW - classification of the clips cut at the violence start (will violence follow?)
Starts from a **finished colab2 run** (trimmed clips, tracks, stories, encoded video windows) and trains/evaluates classifiers: violent clips (cut at the start time T) vs Normal clips.
Every clip is judged on its **last 3 seconds** (text, video windows, geometry), so clip length cannot be read off the input. Normal clips are kept whole.

Inputs: the colab2 output folder. Outputs: `VAW_results/classtrim/` (documents in `text/`, captions in `captions/`, cached features in `features/`, the evaluation in `report/`).
Switch **Layer B** (frame captions by a video-language model) on or off with `USE_CAPTIONS` below. If a class has fewer than 10 clips the report says so instead of printing unstable numbers
(colab2 had 6 clips with a start time at the time of writing: add start times to the CSV and rerun colab2 first).

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/KRITHIKSAI16/Violence-Against-Women.git"
if not os.path.isdir('/content/VAW'):
    !git clone -q {url} /content/VAW
else:
    !git -C /content/VAW pull -q {url} main
%cd /content/VAW
!pip install -q -r colabclasstrim/requirements.txt
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (switch the runtime to a T4 GPU: Runtime > Change runtime type)')
!python -m pytest -q tests/test_class_text_geom.py tests/test_class_metrics_cv.py tests/test_class_pipeline.py

## Settings

In [ ]:
OUT = '/content/drive/Shareddrives/VAW/VAW_results/classtrim'
CURATED = '/content/drive/Shareddrives/VAW/VAW_results/curated_previolence_v2'   # the finished colab2 output folder
USE_CAPTIONS = True      # Layer B on (default). False = geometry text only
flag = '' if USE_CAPTIONS else '--no-captions'
common = f"--out-root {OUT} --curated-root {CURATED} {flag}"
assert os.path.isdir(CURATED), 'the colab2 output folder was not found: run the colab2 notebook first or fix CURATED'

## 1. Layer B: describe a few frames of every clip (GPU; skipped when USE_CAPTIONS is False; resumable)

In [ ]:
!python -m src.classtrim.run --stage captions {common}

## 2. Video features (only the windows colab2 did not already encode; GPU)

In [ ]:
!python -m src.classtrim.run --stage features {common}

## 3. Train, cross-validate, report

In [ ]:
!python -m src.classtrim.run --stage train {common}

## 4. Look at it here

In [ ]:
from IPython.display import HTML, display, IFrame
page = OUT + '/report/index.html'
print('Single-file report (open or download it from Drive):', page)
display(IFrame(page, width='100%', height=900)) if os.path.exists(page) else print('index.html not found: run the train stage first')
print(open(OUT + '/report/report.md', encoding='utf-8').read())